# HW4.a + HW4.5 — BERT fine-tune, vLLM, memory engineering

DATA 266 GPU Assignment II. Repository tag: `hw4-5`.

This notebook must run top to bottom. On Colab: upload this folder (so `src/` sits next to the notebook) or unzip the repo and `cd` into it first.

In [ ]:
# Section 0.1 — personal parameters (must be the first code cell)
SJSU_ID = "019129357"
SID4 = int(SJSU_ID[-4:])          # 9357
SEED = SID4                       # 9357
SLICE = SID4 % 1000               # 357
HP_ID = SID4 % 6                  # 3  Learning-rate-low (unused in this homework)
CLS_A = SID4 % 10                 # 7
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10  # 6

print(f"SJSU_ID={SJSU_ID}")
print(f"SID4={SID4}  SEED={SEED}  SLICE={SLICE}  HP_ID={HP_ID}  CLS_A={CLS_A}  CLS_B={CLS_B}")
print("Twitter working subset: consecutive rows [357, 5356] (5000 rows), 70/15/15 split, seed 9357")

In [ ]:
# Colab only — skip this cell on the RTX 4090.
# Runtime → Change runtime type → GPU (T4).
# The .ipynb is NOT self-contained: it imports src/. Upload a zip of the repo.

import sys, os, glob, subprocess, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import files
    os.chdir("/content")
    if not Path("/content/gpu_assignment2_genai/src").exists() and not Path("/content/src").exists():
        print("Upload the repo zip (NOT just the notebook). Exclude .venv if you can.")
        uploaded = files.upload()  # choose gpu_assignment2_genai.zip
        for name in uploaded:
            if name.endswith(".zip"):
                with zipfile.ZipFile(name) as z:
                    z.extractall("/content")
    for cand in (Path("/content/gpu_assignment2_genai"), Path("/content")):
        if (cand / "src").exists():
            os.chdir(cand)
            break
    print("cwd", os.getcwd(), "src exists", Path("src").exists())
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch", "transformers", "datasets", "accelerate",
        "scikit-learn", "pandas", "numpy", "matplotlib", "tqdm",
        "evaluate", "safetensors", "bitsandbytes",
    ])
    # Real vLLM (Linux/Colab). Ignore failure on CPU runtimes.
    subprocess.call([sys.executable, "-m", "pip", "install", "-q", "vllm"])
else:
    print("Not Colab — using the local src/ tree.")

In [ ]:
import os, sys, random
from pathlib import Path
import numpy as np
import torch

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("repo root:", ROOT)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

from src.config import personal_params_block
from src.gpu_info import capture_system_files, gpu_snapshot
print(personal_params_block())
snap = capture_system_files()
print(snap.get("gpu_name"), snap.get("gpu_uuid"), "VRAM bytes", snap.get("vram_bytes"))

## 4.a BERT sentiment classifier

Data: Hugging Face `tweet_eval` / `sentiment` (negative / neutral / positive).
Tokenizer: `BertTokenizer`. Model: `BertForSequenceClassification` (`bert-base-uncased`, 3 labels).
Working subset: 5,000 consecutive train rows starting at `SLICE=357` → rows **[357, 5356]**.

In [ ]:
from src.data import build_tensor_datasets
from src.bert_engine import TrainConfig, run_training
from src.error_analysis import plot_training_curves, plot_confusion_matrix, worst_class_gallery
from src.config import BASE_BATCH_SIZE, BASE_EPOCHS, BASE_LR, RESULTS_DIR
import json

data = build_tensor_datasets()
print(data["meta"])
print("n train/val/test", len(data["train"]), len(data["val"]), len(data["test"]))

In [ ]:
cfg = TrainConfig(
    batch_size=BASE_BATCH_SIZE,
    epochs=BASE_EPOCHS,
    lr=BASE_LR,
    run_name="hw4a_baseline",
    save=True,
    eval_test=True,
)
result = run_training(cfg)
if result["status"] != "SUCCESS":
    for bs in (4, 2, 1):
        cfg.batch_size = bs
        cfg.run_name = f"hw4a_baseline_bs{bs}"
        result = run_training(cfg)
        if result["status"] == "SUCCESS":
            break
assert result["status"] == "SUCCESS", result.get("status")
print("peak_memory_gb", result["peak_memory_gb"], "wall_s", result["wall_clock_sec"])
print("val stopped improving at epoch", result["stopped_improving_epoch"])

In [ ]:
plot_training_curves(result["epoch_log"], "HW4.a BERT train/val curves", "hw4a_training_curves.png")
plot_confusion_matrix(result["confusion_matrix"])
gallery = worst_class_gallery(result)
print("worst class:", gallery["worst_class"], "recall", gallery["worst_class_recall"])
for ex in gallery["examples"]:
    print("-", ex["true_label"], "->", ex["pred_label"], ":", ex["diagnosis"])
    print("  ", ex["text"][:180])

### Save / reload for custom inference

In [ ]:
from transformers import BertForSequenceClassification, BertTokenizer
from src.config import CHECKPOINTS_DIR, LABEL_NAMES, MAX_LENGTH

ckpt = CHECKPOINTS_DIR / "bert_sentiment_hw4a"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tok = BertTokenizer.from_pretrained(ckpt)
clf = BertForSequenceClassification.from_pretrained(ckpt).to(device).eval()
texts = [
    "I love this class, the GPU labs actually make the architecture click.",
    "This assignment is due tonight and the cluster is down. worst day ever.",
    "The shuttle to campus is at 7:15 as usual.",
]
enc = tok(texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
enc = {k: v.to(device) for k, v in enc.items()}
with torch.no_grad():
    pred = clf(**enc).logits.argmax(-1).cpu().tolist()
for t, y in zip(texts, pred):
    print(LABEL_NAMES[y], ":", t)

## 4.a vLLM vs Hugging Face pipeline

Model: `gpt2` on a 4 GB card, `Qwen/Qwen2.5-0.5B-Instruct` if VRAM ≥ 12 GB.
Custom prompt is in `src/config.py` (`CUSTOM_PROMPT`).
If `import vllm` fails (typical on Windows), the row is recorded as UNAVAILABLE and this cell is re-run on Colab.

In [ ]:
from src.serving import run_serving_suite
serving_rows = run_serving_suite()
import pandas as pd
display(pd.DataFrame(serving_rows)[
    ["engine", "batch_size", "tokens_per_sec", "p50_latency_ms", "p95_latency_ms", "status", "gpu_name"]
])

## HW4.5 Part A — batch scaling to the memory wall

Same HW4.a fine-tune, batch sizes 1, 2, 4, ... until OOM. Repeat this notebook on the second card, then run `scripts/08_make_combined_figures.py`.

In [ ]:
from src.memory_experiments import part_a_batch_scaling
part_a_rows = part_a_batch_scaling(start_batch=1, max_batch=256, epochs=1)
import pandas as pd
display(pd.DataFrame(part_a_rows))

## HW4.5 Part B — memory techniques, one at a time

AMP, gradient checkpointing, gradient accumulation, 8-bit/Adafactor optimiser, then the best combination.

In [ ]:
from src.memory_experiments import part_b_techniques
bs = result["config"]["batch_size"]
part_b_rows = part_b_techniques(baseline_batch=bs, epochs=2)
import pandas as pd
display(pd.DataFrame(part_b_rows))

## HW4.5 Part C — DDP (this machine is 1 GPU)

Instructor two-GPU pairing is not present. This is the same personal-GPU alternative used for HW2.5. For a real two-process job on a multi-GPU node: `torchrun --nproc_per_node=2 scripts/06_part_c_ddp.py`.

In [ ]:
from src.ddp_experiment import run_ddp
ddp = run_ddp(epochs=1, batch_size=8)
print(json.dumps({k: ddp[k] for k in ddp if k != "communication"}, indent=2))
print("communication:", ddp.get("communication"))

## HW4.5 Part D — serving revisited + KV cache vs context length

In [ ]:
from src.serving import kv_cache_sweep
kv_rows = kv_cache_sweep(batch_size=8)
import pandas as pd
display(pd.DataFrame(kv_rows))

## HW4.5 Part E — rebuild METRICS.md

GPU-hours = wall-clock seconds of the chosen full fine-tune / 3600. Do **not** label the summary as an HW5 baseline.

In [ ]:
import runpy
runpy.run_path(str(ROOT / "scripts" / "09_write_metrics.py"), run_name="__main__")
print((ROOT / "METRICS.md").read_text(encoding="utf-8")[:2500])